In [1]:
import pandas as pd
from datetime import datetime
df = pd.read_csv("../data/earthquakes_raw.csv")
#copy of the original df
df_original = df.copy()

In [2]:
df.columns

Index(['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag',
       'magType', 'place', 'status', 'tsunami', 'sig', 'net', 'nst', 'dmin',
       'rms', 'gap', 'type', 'alert', 'felt', 'cdi', 'mmi', 'code', 'types',
       'ids', 'sources'],
      dtype='str')

In [3]:
df.shape

(102437, 26)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 102437 entries, 0 to 102436
Data columns (total 26 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   id         102437 non-null  str    
 1   time       102437 non-null  str    
 2   updated    102437 non-null  str    
 3   latitude   102437 non-null  float64
 4   longitude  102437 non-null  float64
 5   depth_km   102437 non-null  float64
 6   mag        102437 non-null  float64
 7   magType    102437 non-null  str    
 8   place      102437 non-null  str    
 9   status     102437 non-null  str    
 10  tsunami    102437 non-null  int64  
 11  sig        102437 non-null  int64  
 12  net        102437 non-null  str    
 13  nst        89857 non-null   float64
 14  dmin       99292 non-null   float64
 15  rms        102432 non-null  float64
 16  gap        100134 non-null  float64
 17  type       102437 non-null  str    
 18  alert      4069 non-null    str    
 19  felt       15138 non-null   float6

In [5]:
df.isnull().sum()

id               0
time             0
updated          0
latitude         0
longitude        0
depth_km         0
mag              0
magType          0
place            0
status           0
tsunami          0
sig              0
net              0
nst          12580
dmin          3145
rms              5
gap           2303
type             0
alert        98368
felt         87299
cdi          87299
mmi          93258
code             0
types            0
ids              0
sources          0
dtype: int64

In [6]:
# Convert date/time fields (time, updated) to datetime objects.
df['time'] = pd.to_datetime(df['time'])
df['updated'] = pd.to_datetime(df['updated'])
df[["time", "updated"]].dtypes

time       datetime64[us]
updated    datetime64[us]
dtype: object

### 2.Clean Text Fields

In [7]:
# Normalize alert field (if exists) to lowercase.
df['alert'].value_counts(dropna=False)

alert
NaN       98368
green      3901
yellow      122
red          23
orange       23
Name: count, dtype: int64

In [8]:
#Since all the field values are in lowercase, 
#just replaceing the missing values with "none" so the column has no NaN values.
df['alert'] = df['alert'].fillna('none')
df['alert'].value_counts(dropna=False)

alert
none      98368
green      3901
yellow      122
red          23
orange       23
Name: count, dtype: int64

In [9]:
#Use Regex to extract country from place.
#If there is a comma, take everything after the comma. If there is no comma, keep the original value

df['country'] = (
    df['place']
    .str.extract(r',\s*(.+)$')[0]
    .fillna(df['place'])
)

In [10]:
df[['country','place']].head(10)

,country,place
0,Fiji,"235 km E of Levuka, Fiji"
1,south of the Fiji Islands,south of the Fiji Islands
2,Japan,"22 km NE of Hachinohe, Japan"
3,Hawaii,"7 km E of P?hala, Hawaii"
4,Alaska,"188 km SE of Chiniak, Alaska"
5,southern East Pacific Rise,southern East Pacific Rise
6,Alaska,"246 km W of Adak, Alaska"
7,Japan,"8 km NNW of Funaishikawa, Japan"
8,Greece,"5 km SE of Pýrgos, Greece"
9,Japan,"24 km SW of Kushimoto, Japan"


In [11]:
df[['magType', 'status', 'type', 'net', 'sources', 'types']].value_counts()

magType  status    type        net  sources     types                                                                
mb       reviewed  earthquake  us   ,us,        ,origin,phase-data,                                                      61789
ml       reviewed  earthquake  us   ,ak,us,     ,origin,phase-data,                                                       5997
mb       reviewed  earthquake  us   ,us,        ,dyfi,origin,phase-data,                                                  4638
md       reviewed  earthquake  pr   ,us,pr,     ,origin,phase-data,                                                       2369
                                    ,pr,        ,origin,phase-data,                                                       1970
                                                                                                                         ...  
mww      reviewed  earthquake  us   ,us,        ,dyfi,event-sequence,oaf,origin,phase-data,                             

In [12]:
#Ensure all string fields (magType, status, type, net, sources, types) are clean.

for col in ['magType', 'status', 'type', 'net', 'sources', 'types']:
    print(f"\n{col}:")
    print(df[col].head())


magType:
0    mb
1    mb
2    mb
3    ml
4    ml
Name: magType, dtype: str

status:
0    reviewed
1    reviewed
2    reviewed
3    reviewed
4    reviewed
Name: status, dtype: str

type:
0    earthquake
1    earthquake
2    earthquake
3    earthquake
4    earthquake
Name: type, dtype: str

net:
0    us
1    us
2    us
3    hv
4    us
Name: net, dtype: str

sources:
0       ,us,
1       ,us,
2       ,us,
3    ,hv,us,
4    ,ak,us,
Name: sources, dtype: str

types:
0         ,origin,phase-data,
1         ,origin,phase-data,
2         ,origin,phase-data,
3    ,dyfi,origin,phase-data,
4         ,origin,phase-data,
Name: types, dtype: str


In [13]:
#converting the entire type column to lowercase
df['type'] = df['type'].str.strip().str.lower()

In [14]:
#removes the commas at the beginning of the types
df["types"] = (df["types"].str.strip().str.strip(","))

### 3.Clean Numeric Fields

In [15]:
#Before deciding which one to use i checked the distributions of the four partially missing columns:
df[["nst", "dmin", "rms", "gap"]].describe()

,nst,dmin,rms,gap
count,89857.000000,99292.000000,102432.000000,100134.000000
mean,45.716071,3.099479,0.658396,122.774139
std,39.611774,4.387234,0.259593,62.290870
min,0.000000,0.000000,0.000000,8.000000
25%,21.000000,0.826000,0.490000,76.000000
50%,32.000000,1.803000,0.650000,114.000000
75%,56.000000,3.597250,0.820000,156.000000
max,619.000000,62.558000,11.910000,359.000000


In [19]:
df[['nst', 'dmin', 'rms', 'gap']].median()

nst      32.000
dmin      1.802
rms       0.650
gap     114.000
dtype: float64

In [16]:
df["nst"] = df["nst"].fillna(df["nst"].median())
df["dmin"] = df["dmin"].fillna(df["dmin"].median())
df["rms"] = df["rms"].fillna(df["rms"].median())
df["gap"] = df["gap"].fillna(df["gap"].median())
df[["nst", "dmin", "rms", "gap"]].isna().sum()

nst     0
dmin    0
rms     0
gap     0
dtype: int64

In [17]:
df[['felt', 'cdi', 'mmi']].eq(0).sum()

felt    534
cdi     179
mmi     597
dtype: int64

In [18]:
# Adding 0 to the missing values, since it has no information
df[['felt', 'cdi', 'mmi']] = df[['felt', 'cdi', 'mmi']].fillna(0)
df[['felt', 'cdi', 'mmi']].isnull().sum()

felt    0
cdi     0
mmi     0
dtype: int64

In [19]:
# Making sure that any column in the dataset still has missing values.
df.isnull().sum()

id           0
time         0
updated      0
latitude     0
longitude    0
depth_km     0
mag          0
magType      0
place        0
status       0
tsunami      0
sig          0
net          0
nst          0
dmin         0
rms          0
gap          0
type         0
alert        0
felt         0
cdi          0
mmi          0
code         0
types        0
ids          0
sources      0
country      0
dtype: int64

### 4.Add Derived Columns

In [20]:
#Year, month, day, day_of_week from time.
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day
df["day_of_week"] = df["time"].dt.day_name()

In [21]:
df[["time", "year", "month", "day", "day_of_week"]].head()

,time,year,month,day,day_of_week
0,2021-10-31 23:47:49.663,2021,10,31,Sunday
1,2021-10-31 23:39:39.089,2021,10,31,Sunday
2,2021-10-31 23:27:25.244,2021,10,31,Sunday
3,2021-10-31 23:12:16.450,2021,10,31,Sunday
4,2021-10-31 22:25:49.697,2021,10,31,Sunday


In [22]:
df.shape

(102437, 31)

In [23]:
#Shallow/deep earthquake flag based on depth_km.
df["depth_category"] = df["depth_km"].apply(
    lambda x: "shallow" if x < 70 else "deep"
)
df[["depth_km","depth_category"]].head()

,depth_km,depth_category
0,570.45,deep
1,496.53,deep
2,90.56,deep
3,33.22,shallow
4,22.24,shallow


In [24]:
#Strong/destructive flag based on mag thresholds.
df["magnitude_category"] = df["mag"].apply(
    lambda x: "strong" if x < 7.0 else "destructive"
)
df["magnitude_category"].value_counts()

magnitude_category
strong         102366
destructive        71
Name: count, dtype: int64

In [25]:
df.shape

(102437, 33)

In [26]:
df.columns

Index(['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag',
       'magType', 'place', 'status', 'tsunami', 'sig', 'net', 'nst', 'dmin',
       'rms', 'gap', 'type', 'alert', 'felt', 'cdi', 'mmi', 'code', 'types',
       'ids', 'sources', 'country', 'year', 'month', 'day', 'day_of_week',
       'depth_category', 'magnitude_category'],
      dtype='str')

In [27]:
df["magnitude_category"].value_counts()

magnitude_category
strong         102366
destructive        71
Name: count, dtype: int64

In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 102437 entries, 0 to 102436
Data columns (total 33 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   id                  102437 non-null  str           
 1   time                102437 non-null  datetime64[us]
 2   updated             102437 non-null  datetime64[us]
 3   latitude            102437 non-null  float64       
 4   longitude           102437 non-null  float64       
 5   depth_km            102437 non-null  float64       
 6   mag                 102437 non-null  float64       
 7   magType             102437 non-null  str           
 8   place               102437 non-null  str           
 9   status              102437 non-null  str           
 10  tsunami             102437 non-null  int64         
 11  sig                 102437 non-null  int64         
 12  net                 102437 non-null  str           
 13  nst                 102437 non-null  flo

In [29]:
df.isnull().sum()

id                    0
time                  0
updated               0
latitude              0
longitude             0
depth_km              0
mag                   0
magType               0
place                 0
status                0
tsunami               0
sig                   0
net                   0
nst                   0
dmin                  0
rms                   0
gap                   0
type                  0
alert                 0
felt                  0
cdi                   0
mmi                   0
code                  0
types                 0
ids                   0
sources               0
country               0
year                  0
month                 0
day                   0
day_of_week           0
depth_category        0
magnitude_category    0
dtype: int64

In [30]:
#Saving cleaned dataFrame 
df.to_csv("../data/earthquakes_cleaned.csv", index=False)